# 05 — Tourism Intensity

Quantifies tourism presence per grid cell from OSM tourism-related POIs.

**Data source:** Overpass API (OSM `tourism=*`) — fully portable.

**Method:** Single batch Overpass query for all tourism POIs, then BallTree matching to nearest grid cell.

**Output columns:** `cell_id`, `tourism_density`

**Output file:** `csv/05_tourism_intensity.csv`

In [ ]:
# ── Papermill parameters ──────────────────────────────
GRID_CONFIG = "grid.json"

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import os
import math
import hashlib
from sklearn.neighbors import BallTree

os.makedirs("cache", exist_ok=True)

with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]
CELL_AREA_KM2 = (CELL_SIZE_M / 1000) ** 2
QUERY_RADIUS = 500
CSV_DIR = config.get("csv_dir", "csv")

os.makedirs(CSV_DIR, exist_ok=True)

df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
print(f"Loaded {len(df_grid)} grid cells")

BUFFER = 0.015
LAT_MIN = df_grid["cell_lat"].min() - BUFFER
LAT_MAX = df_grid["cell_lat"].max() + BUFFER
LON_MIN = df_grid["cell_lon"].min() - BUFFER
LON_MAX = df_grid["cell_lon"].max() + BUFFER
BBOX = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"

In [ ]:
# ── Overpass batch query helper ───────────────────────

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "grid-finding/1.0 (research project)"}


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            import time
            time.sleep(5 + attempt * 3)
    raise RuntimeError(f"Overpass failed: {last_error}")


print("Helpers ready.")

In [ ]:
# ── Batch query: ALL tourism POIs in study area ───────

TOURISM_VALUES = {"hotel", "hostel", "motel", "guest_house", "museum",
                  "attraction", "viewpoint", "gallery", "artwork",
                  "information", "theme_park", "zoo", "aquarium"}

query = (f'[out:json][timeout:90];\n'
         f'(node["tourism"]({BBOX});\n'
         f' way["tourism"]({BBOX}););\n'
         f'out center tags;')

print("Querying all tourism POIs...")
data = query_overpass_cached(query)

EARTH_RADIUS_M = 6371000
MAX_DIST_M = QUERY_RADIUS

poi_records = []
for el in data.get("elements", []):
    tags = el.get("tags", {})
    tourism_val = tags.get("tourism", "")
    lat = el.get("lat") or (el.get("center", {}) or {}).get("lat")
    lon = el.get("lon") or (el.get("center", {}) or {}).get("lon")
    if lat and lon and tourism_val in TOURISM_VALUES:
        poi_records.append({"lat": float(lat), "lon": float(lon)})

print(f"  Found {len(poi_records)} tourism POIs")

In [ ]:
# ── BallTree: assign each POI to nearest grid cell ───

cell_coords_rad = np.radians(df_grid[["cell_lat", "cell_lon"]].values)
cell_ids = df_grid["cell_id"].tolist()
cell_counts = {c: 0 for c in cell_ids}

if poi_records:
    cell_tree = BallTree(cell_coords_rad, metric="haversine")
    poi_coords = np.radians([[p["lat"], p["lon"]] for p in poi_records])
    distances, indices = cell_tree.query(poi_coords, k=1)

    for dist, idx in zip(distances.flatten(), indices.flatten()):
        if dist * EARTH_RADIUS_M <= MAX_DIST_M:
            cell_counts[cell_ids[idx]] += 1

records = []
for cid in cell_ids:
    tc = cell_counts[cid]
    records.append({
        "cell_id": cid,
        "tourism_density": round(tc / CELL_AREA_KM2, 2),
    })

df_tourism = pd.DataFrame(records)
print(f"Completed: {len(df_tourism)} cells")
print(f"Cells with tourism POIs: {(df_tourism['tourism_density'] > 0).sum()}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/05_tourism_intensity.csv"
df_tourism.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_tourism)} rows x {df_tourism.shape[1]} cols)")
print(df_tourism.describe().round(2).to_string())